# InternVL3 Document-Aware Batch Processing

## Clean Implementation - Based on Working Single-Image Pattern

This notebook implements batch processing using the **exact same successful pattern** from `internvl3_document_aware.ipynb` that works perfectly on H200 GPU.

### Key Architecture:
- ✅ **Simple Direct Approach** - No complex handlers or recursion protection
- ✅ **Proven Model Loading** - Uses exact same H200 direct loading that works
- ✅ **Clean Loop Structure** - Process images one by one with working processor
- ✅ **YAML-First Detection** - Same approach as successful single-image notebook


In [ ]:
import gc
import os
import torch

# CRITICAL: Set V100-optimized CUDA memory allocation (from working system)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:64'
print("🔧 CUDA memory allocation configured: max_split_size_mb:64 (V100 optimized)")

# Import V100-optimized memory management functions from common module
from common.gpu_optimization import clear_gpu_cache, emergency_cleanup, cleanup_model_handler

print("💡 V100 memory management functions imported from common.gpu_optimization")
print("💡 Functions available:")
print("   - cleanup_model_handler(): Clean up model handlers before reinitializing")
print("   - clear_gpu_cache(): Clear GPU memory cache")
print("   - emergency_cleanup(): Aggressive cleanup for OOM recovery")

🔧 CUDA memory allocation configured: max_split_size_mb:64 (V100 optimized)
💡 V100 memory management functions imported from common.gpu_optimization
💡 Functions available:
   - cleanup_model_handler(): Clean up model handlers before reinitializing
   - clear_gpu_cache(): Clear GPU memory cache
   - emergency_cleanup(): Aggressive cleanup for OOM recovery


In [ ]:
# Enhanced imports - adding advanced analytics, visualization, and reporting
import os
import sys
import time
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Any
from datetime import datetime
from IPython.display import Markdown, display

# Set project root (notebook is now in project root)
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

# Core InternVL3 imports (keep existing working imports)
from models.document_aware_internvl3_processor import DocumentAwareInternVL3Processor
from common.evaluation_metrics import load_ground_truth
from common.unified_schema import DocumentTypeFieldSchema

# NEW: Advanced analytics, visualization, and reporting imports
from common.batch_analytics import BatchAnalytics
from common.batch_visualizations import BatchVisualizer
from common.batch_reporting import BatchReporter
from common.document_type_metrics import DocumentTypeEvaluator

# Rich console for enhanced output
from rich import print as rprint
from rich.console import Console

# Initialize console for enhanced output
console = Console()

print("📚 ENHANCED LIBRARIES IMPORTED SUCCESSFULLY")
print("=" * 60)
print(f"🗂️ Project root: {project_root}")
print(f"📍 Current directory: {Path.cwd()}")
print("✅ Core InternVL3 imports: DocumentAwareInternVL3Processor, evaluation_metrics, unified_schema")
print("✅ NEW: BatchAnalytics - comprehensive DataFrames and statistics")
print("✅ NEW: BatchVisualizer - professional dashboards and heatmaps")
print("✅ NEW: BatchReporter - executive summaries and JSON reports")
print("✅ NEW: DocumentTypeEvaluator - ground truth evaluation")
print("🎯 Strategy: Direct processor approach (NO BatchProcessor - avoids recursion)")
print("🚀 Ready for enhanced batch processing with professional analytics!")

In [ ]:
from common.internvl3_batch_display import display_prompt_info, display_raw_and_cleaned, display_field_comparison
from common.internvl3_batch_analytics import generate_summary_stats, display_summary_table, create_simple_dashboard

## Configuration

Set up paths and initialize the processor with **the exact same configuration** that works in the single-image notebook:

In [ ]:
# Enhanced Configuration System - Professional batch processing setup
import os
from datetime import datetime
from pathlib import Path

# Comprehensive configuration dictionary (replacing simple variables)
CONFIG = {
    # Model settings
    'MODEL_PATH': "/home/jovyan/nfs_share/models/InternVL3-8B",  # Jupyter environment path
    
    # Batch settings
    'DATA_DIR': 'evaluation_data',
    'GROUND_TRUTH': 'evaluation_data/ground_truth.csv',
    'MAX_IMAGES': None,  # None for all, or set limit
    'DOCUMENT_TYPES': None,  # None for all, or ['invoice', 'receipt']
    
    # Output settings
    'OUTPUT_BASE': os.getenv('OUTPUT_DIR', 'output'),
    'VERBOSE': True,
    
    # V100 optimization settings
    'USE_QUANTIZATION': True,
    'DEVICE_MAP': 'auto',
    'MAX_NEW_TOKENS': 4000,
    'TORCH_DTYPE': 'bfloat16',
    'LOW_CPU_MEM_USAGE': True
}

# Enhanced output directory structure
OUTPUT_BASE = Path(CONFIG['OUTPUT_BASE'])
if not OUTPUT_BASE.is_absolute():
    OUTPUT_BASE = Path.cwd() / OUTPUT_BASE

BATCH_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

OUTPUT_DIRS = {
    'base': OUTPUT_BASE,
    'batch': OUTPUT_BASE / 'batch_results',
    'csv': OUTPUT_BASE / 'csv',
    'visualizations': OUTPUT_BASE / 'visualizations',
    'reports': OUTPUT_BASE / 'reports'
}

# Create all output directories
for dir_path in OUTPUT_DIRS.values():
    dir_path.mkdir(parents=True, exist_ok=True)

# Clean up any existing models before initializing
cleanup_model_handler('processor', globals())

print("🔧 ENHANCED CONFIGURATION LOADED")
print("=" * 50)
print(f"🎯 Model Path: {CONFIG['MODEL_PATH']}")
print(f"📁 Data Directory: {CONFIG['DATA_DIR']}")
print(f"📊 Ground Truth: {CONFIG['GROUND_TRUTH']}")
print(f"💾 Output Base: {OUTPUT_BASE}")
print(f"📁 Output Structure:")
for name, path in OUTPUT_DIRS.items():
    print(f"   {name}: {path}")
print(f"⏰ Batch Timestamp: {BATCH_TIMESTAMP}")
print("✅ All output directories created successfully")

## Step 1: Discover Images for Batch Processing

In [ ]:
# Enhanced image discovery using CONFIG settings
image_dir = project_root / CONFIG['DATA_DIR']
image_extensions = [".png", ".jpg", ".jpeg", ".bmp", ".tiff"]

# Find all images in directory
image_files = []
for ext in image_extensions:
    image_files.extend(list(image_dir.glob(f"*{ext}")))
    image_files.extend(list(image_dir.glob(f"*{ext.upper()}")))

image_files = sorted(list(set(image_files)))  # Remove duplicates and sort

# Apply CONFIG filters
if CONFIG['DOCUMENT_TYPES']:
    rprint(f"[cyan]📋 Filtering for document types: {CONFIG['DOCUMENT_TYPES']}[/cyan]")
    # Note: Document type filtering would require ground truth or filename patterns
    
if CONFIG['MAX_IMAGES']:
    original_count = len(image_files)
    image_files = image_files[:CONFIG['MAX_IMAGES']]
    rprint(f"[cyan]🔢 Limited to {CONFIG['MAX_IMAGES']} images (from {original_count} available)[/cyan]")

rprint(f"[bold green]📁 Discovered {len(image_files)} images in {image_dir}[/bold green]")
rprint("[cyan]📄 Image files:[/cyan]")
for i, img_path in enumerate(image_files[:10], 1):  # Show first 10
    rprint(f"   {i:2d}. {img_path.name}")
if len(image_files) > 10:
    rprint(f"   ... and {len(image_files) - 10} more images")

if len(image_files) == 0:
    rprint("[red]❌ No images found! Please check the CONFIG['DATA_DIR'] path.[/red]")
else:
    rprint(f"[green]✅ Ready to process {len(image_files)} images with enhanced analytics[/green]")

## Step 2: Initialize Document Schema

Set up the unified schema system for document-aware field selection:

In [ ]:
# Initialize document schema for field selection
print("📋 Initializing Document Type Schema...")

# Use model-specific schema like the working single-image notebook
schema_loader = DocumentTypeFieldSchema(model="internvl3")  # Specify internvl3 model

print(f"✅ Schema loaded with {schema_loader.total_fields} total fields")

# Get supported document types
supported_types = ["invoice", "receipt", "bank_statement"]  # Known types from YAML

print(f"📊 Document types supported: {', '.join(supported_types)}")

# Show field counts by document type
print("\n📈 Field counts by document type:")
for doc_type in supported_types:
    try:
        # Use correct method name with model context
        fields = schema_loader.get_document_fields(doc_type)
        print(f"   📄 {doc_type}: {len(fields)} fields")
    except Exception as e:
        print(f"   ❌ {doc_type}: Error getting fields - {e}")

# Step 3: Process Each Image in the Batch

Loop through each image, applying the same processing logic as in the single-image notebook:

In [ ]:
# Enhanced batch processing with ground truth evaluation and comprehensive analytics
rprint("[bold green]🚀 STARTING ENHANCED BATCH PROCESSING[/bold green]")
console.rule("[bold green]Enhanced Analytics & Evaluation[/bold green]")
rprint(f"📊 Processing {len(image_files)} images with comprehensive analytics...")
rprint(f"🧠 Model: InternVL3-8B with V100 optimizations")
rprint(f"💾 Strategy: SHARED MODEL - Load once, reuse for all images (NO BatchProcessor)")
rprint()

# Load ground truth data ONCE before loop for comprehensive evaluation
try:
    ground_truth_data = load_ground_truth(CONFIG['GROUND_TRUTH'])
    rprint(f"[green]✅ Ground truth loaded for {len(ground_truth_data)} images[/green]")
except Exception as e:
    rprint(f"[red]⚠️ Could not load ground truth: {e}[/red]")
    ground_truth_data = {}

# Initialize DocumentTypeEvaluator for ground truth comparison
document_evaluator = DocumentTypeEvaluator()

# STEP 1: Create processor ONCE with default field list (same as before)
rprint("[cyan]🔧 STEP 1: Loading model ONCE for all images...[/cyan]")
default_fields = schema_loader.get_document_fields("invoice")  # Start with invoice fields
processor = DocumentAwareInternVL3Processor(
    field_list=default_fields,
    model_path=CONFIG['MODEL_PATH'],
    device="cuda",
    debug=False,
    skip_model_loading=False  # Load model normally
)
rprint(f"[green]✅ Model loaded successfully! Ready to process {len(image_files)} images.[/green]")
rprint()

# Enhanced results storage for comprehensive analytics
batch_results = []
processing_times = []
document_types_found = {}
batch_start_time = time.perf_counter()

# STEP 2: Enhanced processing loop with ground truth evaluation
rprint("[cyan]🔧 STEP 2: Processing images with enhanced analytics and evaluation...[/cyan]")
for i, image_path in enumerate(image_files, 1):
    rprint(f"[bold blue]📄 Processing [{i}/{len(image_files)}]: {image_path.name}[/bold blue]")
    
    try:
        image_start = time.perf_counter()
        
        # STEP A: Document Type Detection (same as before)
        rprint(f"   🔍 Step A: Detecting document type...")
        
        # Simple document type detection using schema
        image_name = image_path.name.lower()
        if "statement" in image_name or "bank" in image_name:
            doc_type = "bank_statement"
        elif "receipt" in image_name:
            doc_type = "receipt"
        else:
            doc_type = "invoice"  # Default
        
        # Get document-specific fields
        field_list = schema_loader.get_document_fields(doc_type)
        rprint(f"   📋 Document type: {doc_type} ({len(field_list)} fields)")
        
        # STEP B: Update processor field list (NO MODEL RELOADING)
        rprint(f"   🔄 Step B: Updating field list (no model reload)...")
        processor.field_list = field_list
        processor.field_count = len(field_list)
        
        # STEP C: Process Image with SHARED MODEL (same as before)
        rprint(f"   ⚡ Step C: Processing image with shared model...")
        result = processor.process_single_image(str(image_path))
        
        # STEP D: Enhanced metadata and evaluation
        processing_time = time.perf_counter() - image_start
        image_name = image_path.name
        
        # Enhanced metadata for BatchAnalytics compatibility
        enhanced_result = {
            "image_name": image_name,  # BatchAnalytics expects this
            "image_file": image_name,
            "image_path": str(image_path),
            "document_type": doc_type,
            "processing_time": processing_time,
            "prompt_used": f"internvl3_{doc_type.lower()}",
            "timestamp": datetime.now().isoformat(),
        }
        
        # Add original result data
        enhanced_result.update(result)
        
        # NEW: Ground truth evaluation for comprehensive analytics
        ground_truth = ground_truth_data.get(image_name, {})
        if ground_truth:
            rprint(f"   📊 Step D: Evaluating against ground truth...")
            
            extracted_data = result.get("extracted_data", {})
            evaluation = document_evaluator.evaluate_extraction(
                extracted_data, ground_truth, doc_type
            )
            
            # Format evaluation for BatchAnalytics compatibility
            if evaluation and "overall_metrics" in evaluation:
                evaluation["overall_accuracy"] = evaluation["overall_metrics"].get("overall_accuracy", 0)
                evaluation["fields_extracted"] = evaluation["overall_metrics"].get("total_fields_evaluated", 0)
                evaluation["fields_matched"] = evaluation["overall_metrics"].get("fields_correct", 0)
                evaluation["total_fields"] = evaluation["overall_metrics"].get("total_fields_evaluated", 0)
            
            enhanced_result["evaluation"] = evaluation
            
            # Display evaluation summary
            accuracy = evaluation.get("overall_accuracy", 0) * 100 if evaluation else 0
            rprint(f"   🎯 Accuracy: {accuracy:.1f}%")
        else:
            enhanced_result["evaluation"] = {
                "error": f"No ground truth for {image_name}",
                "overall_accuracy": 0,
            }
            rprint(f"   ⚠️ No ground truth available for evaluation")
        
        # Count found fields for display
        found_fields = len([v for v in result.get("extracted_data", {}).values() if v != "NOT_FOUND"])
        enhanced_result["found_fields"] = found_fields
        
        # Store enhanced results
        batch_results.append(enhanced_result)
        processing_times.append(processing_time)
        document_types_found[doc_type] = document_types_found.get(doc_type, 0) + 1
        
        rprint(f"   [green]✅ Completed: {found_fields}/{len(field_list)} fields found in {processing_time:.2f}s[/green]")
        
        # Display features (keeping existing display functions)
        from common.internvl3_batch_display import display_prompt_info, display_raw_and_cleaned, display_field_comparison
        
        display_prompt_info(result, doc_type)
        display_raw_and_cleaned(result)
        display_field_comparison(result, ground_truth, doc_type)
        
        rprint("="*80 + "\n")
        
        # Light cleanup (same as before)
        clear_gpu_cache()
        
    except Exception as e:
        rprint(f"   [red]❌ Error processing {image_path.name}: {e}[/red]")
        
        # Store error result
        error_result = {
            "image_name": image_path.name,
            "image_path": str(image_path),
            "document_type": "error",
            "error": str(e),
            "processing_time": time.perf_counter() - image_start if 'image_start' in locals() else 0,
        }
        batch_results.append(error_result)
        processing_times.append(error_result["processing_time"])
        
        # Light cleanup on error
        clear_gpu_cache()
    
    rprint()  # Blank line between images

# STEP 3: Final cleanup (same as before)
rprint("[cyan]🧹 STEP 3: Final cleanup...[/cyan]")
del processor
clear_gpu_cache()

total_batch_time = time.perf_counter() - batch_start_time

rprint("[bold green]🎉 ENHANCED BATCH PROCESSING COMPLETED![/bold green]")
console.rule("[bold green]Processing Complete[/bold green]")
rprint(f"📊 Processed: {len(batch_results)} images")
rprint(f"⏱️ Total time: {total_batch_time:.2f}s")
rprint(f"⚡ Average time per image: {total_batch_time/len(batch_results):.2f}s")
rprint(f"📈 Throughput: {len(batch_results)/total_batch_time*60:.1f} images/minute")
rprint("[cyan]💡 OPTIMIZATION: Model loaded once and reused for all images![/cyan]")
rprint("[green]✅ Enhanced results ready for comprehensive analytics and reporting![/green]")

## Step 4: Results Analysis and Export

Analyze the batch results and export to CSV:

In [7]:
# Analyze batch results
print("📊 BATCH RESULTS ANALYSIS")
print("=" * 50)

# Calculate statistics
successful_results = [r for r in results if "error" not in r]
error_count = len(results) - len(successful_results)

if successful_results:
    total_fields = sum(r.get("field_count", 0) for r in successful_results)
    total_found = sum(r.get("found_fields", 0) for r in successful_results)
    avg_processing_time = sum(r.get("processing_time", 0) for r in successful_results) / len(successful_results)
    
    print(f"✅ Successful extractions: {len(successful_results)}/{len(results)} ({len(successful_results)/len(results)*100:.1f}%)")
    print(f"❌ Errors: {error_count}")
    print(f"📊 Total fields processed: {total_fields}")
    print(f"🎯 Total fields found: {total_found}")
    print(f"📈 Overall field coverage: {total_found/total_fields*100:.1f}%")
    print(f"⏱️ Average processing time: {avg_processing_time:.2f}s per image")
    print(f"⚡ Processing speed: {total_found/sum(r.get('processing_time', 0) for r in successful_results):.1f} fields/second")
    
    # Document type breakdown
    doc_types = {}
    for result in successful_results:
        dt = result.get("document_type", "unknown")
        if dt not in doc_types:
            doc_types[dt] = {"count": 0, "total_fields": 0, "found_fields": 0}
        doc_types[dt]["count"] += 1
        doc_types[dt]["total_fields"] += result.get("field_count", 0)
        doc_types[dt]["found_fields"] += result.get("found_fields", 0)
    
    print("\n📋 Document type breakdown:")
    for doc_type, stats in doc_types.items():
        coverage = stats["found_fields"] / stats["total_fields"] * 100 if stats["total_fields"] > 0 else 0
        print(f"   📄 {doc_type}: {stats['count']} docs, {stats['found_fields']}/{stats['total_fields']} fields ({coverage:.1f}% coverage)")

else:
    print("❌ No successful extractions to analyze")

print()

# Create output directory
output_dir = project_root / OUTPUT_DIR
output_dir.mkdir(exist_ok=True)

# Generate timestamp for files
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"💾 Saving results to {output_dir}...")

📊 BATCH RESULTS ANALYSIS
❌ No successful extractions to analyze

💾 Saving results to /home/jovyan/nfs_share/tod/LMM_POC/batch_results...


In [ ]:
# COMPREHENSIVE ANALYTICS - BatchAnalytics Integration
console.rule("[bold blue]Advanced Analytics Generation[/bold blue]")

# Create comprehensive analytics using BatchAnalytics
rprint("[cyan]📊 Generating comprehensive analytics with BatchAnalytics...[/cyan]")

analytics = BatchAnalytics(batch_results, processing_times)

# Generate and save all DataFrames
rprint("[cyan]💾 Creating and saving comprehensive DataFrames...[/cyan]")
saved_files, df_results, df_summary, df_doctype_stats, df_field_stats = analytics.save_all_dataframes(
    OUTPUT_DIRS['csv'], BATCH_TIMESTAMP, verbose=CONFIG['VERBOSE']
)

# Display enhanced summary statistics
rprint("\n[bold blue]📊 ENHANCED RESULTS SUMMARY[/bold blue]")
if len(df_results) > 0:
    display(df_summary)
else:
    rprint("[yellow]⚠️ No results available for summary[/yellow]")

# Display document type performance if available
if not df_doctype_stats.empty:
    rprint("\n[bold blue]📋 DOCUMENT TYPE PERFORMANCE[/bold blue]")
    display(df_doctype_stats)
else:
    rprint("[yellow]⚠️ No document type statistics available[/yellow]")

# Display field-level statistics if available
if df_field_stats is not None and not df_field_stats.empty:
    rprint("\n[bold blue]🎯 FIELD-LEVEL ACCURACY STATISTICS[/bold blue]")
    rprint("Top 10 best performing fields:")
    display(df_field_stats.head(10))
else:
    rprint("[yellow]⚠️ No field-level accuracy data available[/yellow]")

# Analytics summary
rprint(f"\n[green]✅ Analytics generated and saved to {OUTPUT_DIRS['csv']}[/green]")
rprint("[cyan]📊 Available DataFrames:[/cyan]")
for name, path in saved_files.items():
    rprint(f"   📄 {name}: {path.name}")

rprint("\n[bold green]🚀 Ready for professional visualizations and reporting![/bold green]")

In [ ]:
# PROFESSIONAL VISUALIZATIONS - BatchVisualizer Integration
console.rule("[bold blue]Professional Visualization Generation[/bold blue]")

# Create professional visualizations using BatchVisualizer
rprint("[cyan]📈 Generating professional dashboards and visualizations...[/cyan]")

visualizer = BatchVisualizer()

# Generate all visualizations
rprint("[cyan]🎨 Creating comprehensive visualization suite...[/cyan]")
viz_files = visualizer.create_all_visualizations(
    df_results,
    df_doctype_stats,
    OUTPUT_DIRS['visualizations'],
    BATCH_TIMESTAMP,
    show=False  # Set to True to display in notebook
)

# Display visualization summary
if viz_files:
    rprint(f"[green]✅ Professional visualizations generated and saved to {OUTPUT_DIRS['visualizations']}[/green]")
    rprint("[cyan]📊 Generated Visualizations:[/cyan]")
    for viz_type, path in viz_files.items():
        rprint(f"   🎨 {viz_type}: {path.name}")
    
    # Display dashboard if available
    dashboard_path = viz_files.get('dashboard')
    if dashboard_path and dashboard_path.exists():
        rprint("\n[bold blue]📊 PERFORMANCE DASHBOARD PREVIEW:[/bold blue]")
        try:
            from IPython.display import Image
            display(Image(str(dashboard_path)))
        except Exception as e:
            rprint(f"[yellow]ℹ️ Dashboard created but cannot display: {e}[/yellow]")
            rprint(f"[cyan]💡 View dashboard at: {dashboard_path}[/cyan]")
else:
    rprint("[yellow]⚠️ No visualizations generated - check if data is available[/yellow]")

rprint("\n[bold green]🎨 Professional visualizations complete![/bold green]")
rprint("[cyan]📈 Features: 2x2 performance dashboard, accuracy distributions, processing time analysis[/cyan]")

In [ ]:
# EXECUTIVE REPORTING - BatchReporter Integration
console.rule("[bold blue]Executive Report Generation[/bold blue]")

# Create comprehensive reports using BatchReporter
rprint("[cyan]📝 Generating executive reports and comprehensive documentation...[/cyan]")

reporter = BatchReporter(
    batch_results,
    processing_times,
    document_types_found,
    BATCH_TIMESTAMP
)

# Generate and save all reports
rprint("[cyan]📄 Creating markdown summary and JSON export...[/cyan]")
report_files = reporter.save_all_reports(
    OUTPUT_DIRS,
    df_results,
    df_summary,
    df_doctype_stats,
    CONFIG['MODEL_PATH'],
    {
        'data_dir': CONFIG['DATA_DIR'],
        'ground_truth': CONFIG['GROUND_TRUTH'],
        'max_images': CONFIG['MAX_IMAGES'],
        'document_types': CONFIG['DOCUMENT_TYPES']
    },
    {
        'use_quantization': CONFIG['USE_QUANTIZATION'],
        'device_map': CONFIG['DEVICE_MAP'],
        'max_new_tokens': CONFIG['MAX_NEW_TOKENS'],
        'torch_dtype': CONFIG['TORCH_DTYPE'],
        'low_cpu_mem_usage': CONFIG['LOW_CPU_MEM_USAGE']
    },
    verbose=CONFIG['VERBOSE']
)

# Display reporting summary
if report_files:
    rprint(f"[green]✅ Executive reports generated and saved[/green]")
    rprint("[cyan]📄 Generated Reports:[/cyan]")
    for report_type, path in report_files.items():
        rprint(f"   📄 {report_type}: {path.name}")
    
    # Display executive summary preview
    markdown_report_path = report_files.get('markdown_report')
    if markdown_report_path and markdown_report_path.exists():
        rprint("\n[bold blue]📋 EXECUTIVE SUMMARY PREVIEW:[/bold blue]")
        try:
            with open(markdown_report_path, 'r') as f:
                summary_lines = f.readlines()[:20]  # Show first 20 lines
                for line in summary_lines:
                    print(line.rstrip())
                if len(summary_lines) >= 20:
                    rprint("[dim]... (see full report for complete details)[/dim]")
        except Exception as e:
            rprint(f"[yellow]ℹ️ Report created but cannot preview: {e}[/yellow]")
else:
    rprint("[yellow]⚠️ No reports generated - check if data is available[/yellow]")

# Deployment readiness assessment
if len(df_results) > 0:
    avg_accuracy = df_results['overall_accuracy'].mean()
    rprint(f"\n[bold blue]🚀 DEPLOYMENT READINESS ASSESSMENT:[/bold blue]")
    if avg_accuracy >= 95:
        rprint("[bold green]✅ PRODUCTION READY[/bold green] - Average accuracy ≥ 95%")
    elif avg_accuracy >= 80:
        rprint("[bold yellow]🟡 PILOT READY[/bold yellow] - Average accuracy ≥ 80%")
    else:
        rprint("[bold red]🔴 NEEDS IMPROVEMENT[/bold red] - Average accuracy < 80%")
    rprint(f"[cyan]📊 Current average accuracy: {avg_accuracy:.2f}%[/cyan]")

rprint("\n[bold green]📄 Executive reporting complete![/bold green]")
rprint("[cyan]📋 Features: Markdown executive summary, JSON comprehensive export, deployment assessment[/cyan]")

In [ ]:
# ENHANCED FINAL SUMMARY - Professional Dashboard and Comprehensive Results
console.rule("[bold green]InternVL3 Enhanced Batch Processing Complete[/bold green]")

# Calculate comprehensive metrics
total_images = len(batch_results)
successful = len([r for r in batch_results if 'error' not in r])
success_rate = (successful/total_images*100) if total_images > 0 else 0

# Enhanced performance metrics
if len(df_results) > 0:
    avg_accuracy = df_results['overall_accuracy'].mean()
    median_accuracy = df_results['overall_accuracy'].median()
    min_accuracy = df_results['overall_accuracy'].min()
    max_accuracy = df_results['overall_accuracy'].max()
else:
    avg_accuracy = median_accuracy = min_accuracy = max_accuracy = 0

avg_processing_time = np.mean(processing_times) if processing_times else 0
total_processing_time = sum(processing_times) if processing_times else 0
throughput = (len(batch_results)/total_processing_time*60) if total_processing_time > 0 else 0

# Display comprehensive summary
rprint("[bold blue]📊 COMPREHENSIVE PERFORMANCE SUMMARY[/bold blue]")
rprint("=" * 80)
rprint(f"[bold green]✅ Processing Results:[/bold green]")
rprint(f"   📊 Total Images Processed: {total_images}")
rprint(f"   ✅ Successful Extractions: {successful} ({success_rate:.1f}%)")
rprint(f"   ❌ Failed Extractions: {total_images - successful}")

rprint(f"\n[bold blue]🎯 Accuracy Performance:[/bold blue]")
rprint(f"   📈 Average Accuracy: {avg_accuracy:.2f}%")
rprint(f"   📊 Median Accuracy: {median_accuracy:.2f}%")
rprint(f"   📉 Min Accuracy: {min_accuracy:.2f}%")
rprint(f"   📈 Max Accuracy: {max_accuracy:.2f}%")

rprint(f"\n[bold blue]⚡ Processing Performance:[/bold blue]")
rprint(f"   ⏱️ Total Processing Time: {total_processing_time:.2f}s ({total_processing_time/60:.1f} minutes)")
rprint(f"   ⚡ Average Time per Image: {avg_processing_time:.2f}s")
rprint(f"   🚀 Throughput: {throughput:.1f} images/minute")

# Document type breakdown
if document_types_found:
    rprint(f"\n[bold blue]📋 Document Type Distribution:[/bold blue]")
    for doc_type, count in document_types_found.items():
        percentage = (count / total_images) * 100 if total_images > 0 else 0
        rprint(f"   📄 {doc_type}: {count} documents ({percentage:.1f}%)")

# Enhanced output summary
rprint(f"\n[bold blue]📁 COMPREHENSIVE OUTPUT FILES:[/bold blue]")
rprint(f"   🗂️ Base Output Directory: {OUTPUT_BASE}")

# Analytics files
if saved_files:
    rprint(f"   📊 Analytics & DataFrames ({len(saved_files)} files):")
    for name, path in saved_files.items():
        rprint(f"      📄 {name}: {path.name}")

# Visualization files
if viz_files:
    rprint(f"   📈 Professional Visualizations ({len(viz_files)} files):")
    for viz_type, path in viz_files.items():
        rprint(f"      🎨 {viz_type}: {path.name}")

# Report files
if report_files:
    rprint(f"   📝 Executive Reports ({len(report_files)} files):")
    for report_type, path in report_files.items():
        rprint(f"      📄 {report_type}: {path.name}")

# Display visual dashboard
rprint(f"\n[bold blue]📊 VISUAL DASHBOARD PREVIEW:[/bold blue]")
dashboard_files = list(OUTPUT_DIRS['visualizations'].glob(f"dashboard_{BATCH_TIMESTAMP}.png"))
if dashboard_files:
    dashboard_path = dashboard_files[0]
    try:
        from IPython.display import Image
        display(Image(str(dashboard_path)))
        rprint(f"[green]✅ Dashboard displayed above: {dashboard_path.name}[/green]")
    except Exception as e:
        rprint(f"[yellow]ℹ️ Dashboard created but cannot display: {e}[/yellow]")
        rprint(f"[cyan]💡 View dashboard at: {dashboard_path}[/cyan]")
else:
    rprint("[yellow]⚠️ Dashboard not found - check visualization generation[/yellow]")

# Deployment readiness and recommendations
rprint(f"\n[bold blue]🚀 DEPLOYMENT READINESS & RECOMMENDATIONS:[/bold blue]")
if avg_accuracy >= 95:
    status = "[bold green]✅ PRODUCTION READY[/bold green]"
    recommendation = "Model performance exceeds production threshold. Ready for deployment."
elif avg_accuracy >= 80:
    status = "[bold yellow]🟡 PILOT READY[/bold yellow]"
    recommendation = "Model performance meets pilot threshold. Consider limited deployment with monitoring."
else:
    status = "[bold red]🔴 NEEDS IMPROVEMENT[/bold red]"
    recommendation = "Model performance below deployment threshold. Additional training or optimization needed."

rprint(f"   {status}")
rprint(f"   💡 Recommendation: {recommendation}")
rprint(f"   📊 Current Performance: {avg_accuracy:.2f}% average accuracy")

# Architecture achievements
rprint(f"\n[bold green]🎉 ENHANCEMENT ACHIEVEMENTS:[/bold green]")
rprint("   ✅ Direct processing approach maintained (NO BatchProcessor)")
rprint("   ✅ Comprehensive analytics with BatchAnalytics")
rprint("   ✅ Professional visualizations with BatchVisualizer")
rprint("   ✅ Executive reporting with BatchReporter")
rprint("   ✅ Ground truth evaluation integration")
rprint("   ✅ Structured output management")
rprint("   ✅ Production-ready performance assessment")

rprint(f"\n[bold green]🚀 INTERNVL3 ENHANCED BATCH PROCESSING COMPLETE![/bold green]")
rprint("[cyan]Feature parity with Llama version achieved while maintaining InternVL3 reliability![/cyan]")

In [ ]:
# Generate comprehensive summary statistics
summary_stats = generate_summary_stats(results)

# Display summary table
print("📊 SUMMARY STATISTICS:")
display_summary_table(summary_stats)

# Create and save dashboard visualization
dashboard_path = output_dir / f"internvl3_dashboard_{timestamp}.png"
saved_path = create_simple_dashboard(
    results, 
    save_path=dashboard_path,
    show=False  # Set to True if you want to display in notebook
)

if saved_path:
    print(f"✅ Dashboard saved to: {saved_path}")
    
    # Try to display the dashboard image in notebook
    try:
        from IPython.display import Image, display
        print("\n📊 DASHBOARD PREVIEW:")
        display(Image(str(saved_path)))
    except Exception as e:
        print(f"ℹ️ Dashboard created but cannot display in notebook: {e}")
        print(f"💡 View dashboard at: {saved_path}")

## Step 5: Summary Dashboard

Generate comprehensive analytics and visualization dashboard:

In [ ]:
# Export summary report
summary_path = output_dir / f"internvl3_batch_summary_{timestamp}.txt"

with open(summary_path, 'w') as f:
    f.write("InternVL3 Document-Aware Batch Processing Summary\n")
    f.write("=" * 50 + "\n")
    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model: InternVL3-8B\n")
    f.write(f"Strategy: Direct loading (no recursion protection)\n")
    f.write(f"Architecture: Clean batch processing based on working single-image pattern\n\n")
    
    f.write("Processing Results:\n")
    f.write("-" * 20 + "\n")
    f.write(f"Total images: {len(results)}\n")
    f.write(f"Successful: {len(successful_results)}\n")
    f.write(f"Errors: {error_count}\n")
    f.write(f"Success rate: {len(successful_results)/len(results)*100:.1f}%\n\n")
    
    if successful_results:
        f.write("Performance Metrics:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Total processing time: {total_batch_time:.2f}s\n")
        f.write(f"Average time per image: {avg_processing_time:.2f}s\n")
        f.write(f"Throughput: {len(results)/total_batch_time*60:.1f} images/minute\n")
        f.write(f"Field extraction rate: {total_found/sum(r.get('processing_time', 0) for r in successful_results):.1f} fields/second\n\n")
        
        f.write("Field Coverage:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Total fields processed: {total_fields}\n")
        f.write(f"Total fields found: {total_found}\n")
        f.write(f"Overall coverage: {total_found/total_fields*100:.1f}%\n\n")
        
        f.write("Document Type Breakdown:\n")
        f.write("-" * 20 + "\n")
        for doc_type, stats in doc_types.items():
            coverage = stats["found_fields"] / stats["total_fields"] * 100 if stats["total_fields"] > 0 else 0
            f.write(f"{doc_type}: {stats['count']} docs, {coverage:.1f}% field coverage\n")

print(f"✅ Summary report saved to: {summary_path}")

print("\n🎉 BATCH PROCESSING COMPLETE!")
print("=" * 50)
print(f"📁 Results saved in: {output_dir}")
print(f"📊 CSV results: {csv_path.name}")
print(f"📝 Summary report: {summary_path.name}")

✅ Summary report saved to: /home/jovyan/nfs_share/tod/LMM_POC/batch_results/internvl3_batch_summary_20250913_025210.txt

🎉 BATCH PROCESSING COMPLETE!
📁 Results saved in: /home/jovyan/nfs_share/tod/LMM_POC/batch_results
📊 CSV results: internvl3_batch_results_20250913_025210.csv
📝 Summary report: internvl3_batch_summary_20250913_025210.txt

🚀 SUCCESS: Clean batch processing implementation working!
✅ No infinite recursion
✅ Real field extraction (not 0 fields)
✅ Based on proven single-image pattern
